# Supplementary Tables S2 (S22) and S3 (S18d) — manuscript v57

Computed only from committed result files:
* **Table S2** — `benchmarks/real_flow/s22/scores.csv` (per seed × cohort statistics), gate definitions and outcomes from
  `benchmarks/real_flow/s22/RESULT_S22.md` (re-derived from the seed-0 rows and asserted equal to the reported pass flags and to `summary.json`).
* **Table S3** — `benchmarks/perturb_dose/gates.json` (per screen × normalisation × tertile), with medians re-derived from
  `benchmarks/perturb_dose/capture_{screen}_{norm}.csv` and the one-sided Wilcoxon / Mann–Whitney p-values recomputed with SciPy and asserted equal.

Outputs: `manuscript/tables/TableS2_S22.{csv,md}`, `manuscript/tables/TableS3_S18d.{csv,md}`.

In [1]:
from _style import *
import json as _json, re
from scipy.stats import wilcoxon, mannwhitneyu

prov = Provenance("TABLES", notebook="notebooks/tables_v57.ipynb", kind="report")
TABDIR = REPO / "manuscript" / "tables"
TABDIR.mkdir(parents=True, exist_ok=True)
sc = pd.read_csv(prov.use("benchmarks/real_flow/s22/scores.csv", "S2", "S22"))
s22md = prov.use("benchmarks/real_flow/s22/RESULT_S22.md", "S2", "S22").read_text()
s22sum = _json.loads(prov.use("benchmarks/real_flow/s22/summary.json", "S2", "S22").read_text())
s22pre = _json.loads(prov.use("benchmarks/real_flow/s22/preregistration.json", "S2", "S22").read_text())
s22fit = {sd: _json.loads(prov.use(f"benchmarks/real_flow/s22/fit_seed{sd}.json", "S2", "S22").read_text()) for sd in [0, 1, 2]}
S22_DIM = int(np.load(prov.use("benchmarks/real_flow/s22/cells_seed0.npz", "S2", "S22"))["eigenvalues"].shape[1])
S22_PER_DAY = {s22fit[sd]["n_training_cells"] // 3 for sd in s22fit}; assert len(S22_PER_DAY) == 1; S22_PER_DAY = S22_PER_DAY.pop()
gd = _json.loads(prov.use("benchmarks/perturb_dose/gates.json", "S3", "S18d").read_text())
CAP = {(s, n): pd.read_csv(prov.use(f"benchmarks/perturb_dose/capture_{s}_{n}.csv", "S3", "S18d"))
       for s in ["rpe1", "k562"] for n in ["N1", "N2"]}
_nd = {_json.loads(prov.use(f"benchmarks/perturb_dose/capture_{s}_{n}.json", "S3", "S18d").read_text())["null_definition"] for s in ["rpe1", "k562"] for n in ["N1", "N2"]}
assert len(_nd) == 1; N_NULL = int(re.match(r"(\d+) across-T medians", _nd.pop()).group(1))

def fmt(x, nd=3):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "—"
    return f2(x, nd).replace("-", "−")
def fp(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "—"
    return f"{p:.3g}"
def esc(s):
    return str(s).replace("|", "\\|")

## Table S2 — S22: LARRY clonal fate coherence vs score-J stiffness

In [2]:
COHORTS = [c for c in ["pooled", "undifferentiated", "pooled_without_tied", "t2", "t4"] if c in set(sc.cohort)]
s2 = sc[sc.cohort.isin(COHORTS)].copy()
s2["cohort"] = pd.Categorical(s2.cohort, COHORTS, ordered=True)
s2 = s2.sort_values(["seed", "cohort"]).reset_index(drop=True)
s2["cohort"] = s2.cohort.astype(str)
assert sorted(s2.seed.unique()) == [0, 1, 2]
# gated rows: seed 0 pooled (L1) and seed 0 undifferentiated (L2, L3), as flagged in scores.csv
assert set(map(tuple, s2[s2.gated][["seed", "cohort"]].values)) == {(0, "pooled"), (0, "undifferentiated")}
g0 = s2[s2.seed == 0].set_index("cohort")
L = {"L1": bool(g0.loc["pooled", "rho_lambda1_H"] > 0 and g0.loc["pooled", "p_permutation_positive"] < 0.01),
     "L2": bool(g0.loc["undifferentiated", "rho_lambda1_H"] > 0 and g0.loc["undifferentiated", "p_permutation_positive"] < 0.01),
     "L3": bool(abs(g0.loc["undifferentiated", "rho_lambda1_H"]) > abs(g0.loc["undifferentiated", "rho_density_H"])
                and abs(g0.loc["undifferentiated", "rho_lambda1_H"]) > abs(g0.loc["undifferentiated", "rho_centroid_distance_H"])
                and g0.loc["undifferentiated", "partial_rho_given_density"] > 0)}
GDEF = {}
for gname in ["L1", "L2", "L3"]:
    m = re.search(rf"^\| {gname} \| (.*?) \| (.*?) \| (True|False) \|$", s22md, re.M)
    GDEF[gname] = m.group(1)
    assert (m.group(3) == "True") == L[gname], gname
assert s22sum["gate_L1_pooled"]["passed"] == L["L1"] and s22sum["gate_L2_undifferentiated"]["passed"] == L["L2"] and s22sum["gate_L3_baselines"]["passed"] == L["L3"]
s2["gate"] = ""
s2.loc[(s2.seed == 0) & (s2.cohort == "pooled"), "gate"] = "L1"
s2.loc[(s2.seed == 0) & (s2.cohort == "undifferentiated"), "gate"] = "L2, L3"
s2["gate_pass"] = s2.gate.map(lambda g: "" if not g else str(all(L[x.strip()] for x in g.split(","))))
out2 = s2[["seed", "cohort", "gated", "gate", "gate_pass", "n_pairs", "n_tied_pairs", "n_unique_clones", "rho_lambda1_H", "p_permutation_positive",
           "rho_density_H", "rho_centroid_distance_H", "partial_rho_given_density"]]
out2.to_csv(TABDIR / "TableS2_S22.csv", index=False)
for k_, v_ in L.items():
    prov.value(f"S2_{k_}_pass", v_)
prov.value("S2_n_rows", int(len(out2)))
print(L); out2

{'L1': False, 'L2': False, 'L3': False}


,seed,cohort,gated,gate,gate_pass,n_pairs,n_tied_pairs,n_unique_clones,rho_lambda1_H,p_permutation_positive,rho_density_H,rho_centroid_distance_H,partial_rho_given_density
0,0,pooled,True,L1,False,1818,67,1694,-0.002643,1.000000,0.078052,0.058033,0.011456
1,0,undifferentiated,True,"L2, L3",False,1401,0,1317,-0.101161,1.000000,0.076324,0.221025,-0.077741
2,0,pooled_without_tied,False,,,1751,0,1631,0.001412,0.999001,0.082234,0.056209,0.016745
3,0,t2,False,,,232,3,232,-0.098754,0.955045,0.194652,0.114087,0.033176
4,0,t4,False,,,1586,64,1586,0.012317,0.999001,0.082307,0.031811,0.022460
5,1,pooled,False,,,1818,67,1694,-0.018425,0.999001,0.078052,0.058033,-0.009025
6,1,undifferentiated,False,,,1401,0,1317,-0.110547,1.000000,0.076324,0.221025,-0.091060
7,1,pooled_without_tied,False,,,1751,0,1631,-0.009260,1.000000,0.082234,0.056209,0.001167
8,1,t2,False,,,232,3,232,-0.166302,0.995005,0.194652,0.114087,-0.048274
9,1,t4,False,,,1586,64,1586,0.004834,1.000000,0.082307,0.031811,0.007836


In [3]:
CLAB = {"pooled": "pooled", "undifferentiated": "Undifferentiated", "pooled_without_tied": "pooled, tied pairs removed", "t2": "day 2 → 4", "t4": "day 4 → 6"}
n0 = g0.loc["pooled"]
T2 = ["**Supplementary Table S2 | S22: score-J stiffness vs LARRY sister-cell fate coherence.** "
      f"Spearman ρ between a clone's median score-J λ₁ on day *t* and the Shannon entropy H of its cell types on day *t* + 2, for "
      f"{int(n0.n_pairs):,} clone/time pairs ({int(n0.n_unique_clones):,} distinct clones; {int(n0.n_tied_pairs)} pairs with tied modal types). "
      f"Score operator refitted with seeds {min(s22pre['score_config']['seeds'])}–{max(s22pre['score_config']['seeds'])} (energy network, exact-Laplacian score matching, "
      f"LARRY PCA-{S22_DIM}, {S22_PER_DAY:,} cells per day). "
      f"p: one-sided (ρ > 0) permutation p-value, {s22pre['permutations']:,} within-stratum permutations. Baselines: ρ of local density and of distance to the day-2 "
      "Undifferentiated centroid with H; partial ρ: λ₁ vs H given density. **Pre-registered** gates use seed 0 only (rows marked L1–L3); "
      "seeds 1–2, the tie-free pooled cohort and the per-day cohorts are **report only**. Source: `benchmarks/real_flow/s22/scores.csv`.", "",
      "| seed | cohort | gate | n pairs | n tied | ρ(λ₁, H) | p (perm.) | ρ density | ρ centroid distance | partial ρ given density |",
      "|---|---|---|---:|---:|---:|---:|---:|---:|---:|"]
for _, r in out2.iterrows():
    gate = f"**{r.gate}** ({'pass' if r.gate_pass == 'True' else 'fail'})" if r.gate else "report only"
    T2.append(f"| {r.seed} | {CLAB[r.cohort]} | {gate} | {int(r.n_pairs):,} | {int(r.n_tied_pairs)} | {fmt(r.rho_lambda1_H)} | {fp(r.p_permutation_positive)} | "
              f"{fmt(r.rho_density_H)} | {fmt(r.rho_centroid_distance_H)} | {fmt(r.partial_rho_given_density)} |")
T2 += ["", "Gate outcomes (seed 0; definitions as fixed in the task, from `RESULT_S22.md`):", "",
       "| gate | condition | outcome |", "|---|---|---|"]
for gname in ["L1", "L2", "L3"]:
    T2.append(f"| {gname} | {esc(GDEF[gname])} | {'pass' if L[gname] else 'fail'} |")
(TABDIR / "TableS2_S22.md").write_text("\n".join(T2) + "\n")
print("\n".join(T2))

**Supplementary Table S2 | S22: score-J stiffness vs LARRY sister-cell fate coherence.** Spearman ρ between a clone's median score-J λ₁ on day *t* and the Shannon entropy H of its cell types on day *t* + 2, for 1,818 clone/time pairs (1,694 distinct clones; 67 pairs with tied modal types). Score operator refitted with seeds 0–2 (energy network, exact-Laplacian score matching, LARRY PCA-30, 4,000 cells per day). p: one-sided (ρ > 0) permutation p-value, 1,000 within-stratum permutations. Baselines: ρ of local density and of distance to the day-2 Undifferentiated centroid with H; partial ρ: λ₁ vs H given density. **Pre-registered** gates use seed 0 only (rows marked L1–L3); seeds 1–2, the tie-free pooled cohort and the per-day cohorts are **report only**. Source: `benchmarks/real_flow/s22/scores.csv`.

| seed | cohort | gate | n pairs | n tied | ρ(λ₁, H) | p (perm.) | ρ density | ρ centroid distance | partial ρ given density |
|---|---|---|---:|---:|---:|---:|---:|---:|---:|
| 0 | pooled

## Table S3 — S18d: dose response of score-J capture (RPE1 primary, K562 report only)

In [4]:
rows = []
for s in ["rpe1", "k562"]:
    for n in ["N1", "N2"]:
        cap_ = CAP[(s, n)]
        for t in ["weak", "medium", "strong"]:
            e = gd["comparisons"][s][n][t]
            d = cap_[cap_.tertile == t]; T_, M_ = d[d.group_T], d[d.group_M]
            # re-derive from the per-target capture table
            assert len(T_) == e["n_T"] and len(M_) == e["n_M"]
            assert abs(T_.capture_S_J.median() - e["median_S_J_T"]) < 1e-12 and abs(T_.capture_S_C.median() - e["median_S_C_T"]) < 1e-12
            assert abs(M_.capture_S_J.median() - e["median_S_J_M"]) < 1e-12
            pw = wilcoxon(T_.capture_S_J.values, T_.capture_S_C.values, alternative="greater").pvalue
            pm = mannwhitneyu(T_.capture_S_J.values, M_.capture_S_J.values, alternative="greater").pvalue
            assert abs(pw - e["wilcoxon"]["p_one_sided_greater"]) < 1e-9 and abs(pm - e["mannwhitney"]["p_one_sided_greater"]) < 1e-9, (s, n, t)
            # criteria as fixed in the task: D1 median > null95; D2 Wilcoxon p < 0.05; D3 Mann-Whitney p < 0.05
            assert (e["median_S_J_T"] > e["null_across_T_median_p95"]) == e["D1_criterion_met"]
            assert (pw < 0.05) == e["D2_criterion_met"] and (pm < 0.05) == e["D3_criterion_met"]
            prim = s == gd["primary_screen"] and t == "weak"
            rows.append(dict(screen=s.upper(), normalisation=n, tertile=t,
                             status=("pre-registered (" + gd["outcome"] + ")") if prim else "report only",
                             n_T=e["n_T"], n_M=e["n_M"], T_targets=";".join(e["T_targets"]),
                             median_S_J_T=e["median_S_J_T"], median_S_C_T=e["median_S_C_T"], median_S_J_M=e["median_S_J_M"],
                             null95_T_median=e["null_across_T_median_p95"],
                             D1=e["D1_criterion_met"], D2=e["D2_criterion_met"], D3=e["D3_criterion_met"],
                             p_wilcoxon_SJ_gt_SC=pw, p_mannwhitney_T_gt_M=pm))
out3 = pd.DataFrame(rows)
out3.to_csv(TABDIR / "TableS3_S18d.csv", index=False)
assert gd["underpowered"] and gd["n_T"]["rpe1"] < gd["underpowered_threshold_n_T"]
prov.value("S3_outcome", gd["outcome"]); prov.value("S3_n_T_rpe1", gd["n_T"]["rpe1"]); prov.value("S3_n_T_k562", gd["n_T"]["k562"])
prov.value("S3_threshold_n_T", gd["underpowered_threshold_n_T"]); prov.value("S3_n_rows", int(len(out3)))
out3.drop(columns="T_targets")

,screen,normalisation,tertile,status,n_T,n_M,median_S_J_T,median_S_C_T,median_S_J_M,null95_T_median,D1,D2,D3,p_wilcoxon_SJ_gt_SC,p_mannwhitney_T_gt_M
0,RPE1,N1,weak,pre-registered (underpowered),1,5,0.258760,0.131628,0.229736,0.444391,False,False,False,0.500000,0.500000
1,RPE1,N1,medium,report only,1,5,0.190663,0.281137,0.199846,0.503402,False,False,False,1.000000,0.666667
2,RPE1,N1,strong,report only,1,5,0.200433,0.307485,0.234372,0.518021,False,False,False,1.000000,0.666667
3,RPE1,N2,weak,pre-registered (underpowered),1,5,0.268036,0.617288,0.418935,0.641648,False,False,False,1.000000,0.833333
4,RPE1,N2,medium,report only,1,5,0.286172,0.767472,0.504055,0.706730,False,False,False,1.000000,1.000000
5,RPE1,N2,strong,report only,1,5,0.299721,0.697071,0.442478,0.671849,False,False,False,1.000000,1.000000
6,K562,N1,weak,report only,14,38,0.422450,0.346701,0.290762,0.439834,False,False,True,0.270813,0.002137
7,K562,N1,medium,report only,14,38,0.410317,0.362054,0.264744,0.482106,False,False,True,0.357422,0.000172
8,K562,N1,strong,report only,14,38,0.452115,0.337918,0.285280,0.484242,False,True,True,0.012268,0.000050
9,K562,N2,weak,report only,14,38,0.339895,0.202935,0.222440,0.386790,False,False,True,0.086304,0.011943


In [5]:
YN = lambda b: "yes" if b else "no"
T3 = ["**Supplementary Table S3 | S18d: dose response of score-J capture of knockdown displacements (Replogle CRISPRi).** "
      "For each target, cells were split into knockdown tertiles; capture = fraction of the tertile's centroid shift from all controls "
      "(PCA-20 of each normalisation) lying in the three control score-J soft directions (S_J) or the three top control-covariance directions (S_C). "
      "T: transcription-machinery targets; M: mitochondrial/translation targets (medians over targets). null95: 95th percentile of the across-T median "
      f"capture under {N_NULL:,} covariance-matched random 3-dimensional subspaces (one shared basis per draw). Criteria (fixed in the task): D1 median S_J(T) > null95; "
      "D2 S_J > S_C over T, one-sided paired Wilcoxon p < 0.05; D3 S_J(T) > S_J(M), one-sided Mann–Whitney p < 0.05. N1: depth-projected log-normalisation; "
      "N2: analytic Pearson residuals. **Pre-registered:** RPE1, weak tertile, D1–D3 each required in both N1 and N2, with fewer than "
      f"{gd['underpowered_threshold_n_T']} qualifying T targets reported as underpowered. RPE1 had n_T = {gd['n_T']['rpe1']} "
      f"({', '.join(gd['comparisons']['rpe1']['N1']['weak']['T_targets'])}) and n_M = {gd['n_M']['rpe1']}: outcome **{gd['outcome']}**, no gate claimed. "
      f"K562 (n_T = {gd['n_T']['k562']}, n_M = {gd['n_M']['k562']}) and the medium and strong tertiles are **report only**. "
      "Source: `benchmarks/perturb_dose/gates.json` (medians re-derived from `capture_{screen}_{norm}.csv`; p-values recomputed).", "",
      "| screen | norm. | tertile | status | n T / M | median S_J(T) | median S_C(T) | null95 | D1 | D2 | D3 | p Wilcoxon (S_J > S_C) | p Mann–Whitney (T > M) |",
      "|---|---|---|---|---:|---:|---:|---:|---|---|---|---:|---:|"]
for _, r in out3.iterrows():
    T3.append(f"| {r.screen} | {r.normalisation} | {r.tertile} | {r.status} | {r.n_T} / {r.n_M} | {fmt(r.median_S_J_T)} | {fmt(r.median_S_C_T)} | "
              f"{fmt(r.null95_T_median)} | {YN(r.D1)} | {YN(r.D2)} | {YN(r.D3)} | {fp(r.p_wilcoxon_SJ_gt_SC)} | {fp(r.p_mannwhitney_T_gt_M)} |")
T3 += ["", f"Overall outcome (pre-registered RPE1): **{gd['outcome']}** (D1–D3 status: "
       + ", ".join(f"{k_} {v_['status']}" for k_, v_ in gd["gates"].items()) + ")."]
(TABDIR / "TableS3_S18d.md").write_text("\n".join(T3) + "\n")
prov.panel("S2", "Supplementary Table S2 (S22): manuscript/tables/TableS2_S22.{csv,md}")
prov.panel("S3", "Supplementary Table S3 (S18d): manuscript/tables/TableS3_S18d.{csv,md}")
prov.write()
print("\n".join(T3))

**Supplementary Table S3 | S18d: dose response of score-J capture of knockdown displacements (Replogle CRISPRi).** For each target, cells were split into knockdown tertiles; capture = fraction of the tertile's centroid shift from all controls (PCA-20 of each normalisation) lying in the three control score-J soft directions (S_J) or the three top control-covariance directions (S_C). T: transcription-machinery targets; M: mitochondrial/translation targets (medians over targets). null95: 95th percentile of the across-T median capture under 10,000 covariance-matched random 3-dimensional subspaces (one shared basis per draw). Criteria (fixed in the task): D1 median S_J(T) > null95; D2 S_J > S_C over T, one-sided paired Wilcoxon p < 0.05; D3 S_J(T) > S_J(M), one-sided Mann–Whitney p < 0.05. N1: depth-projected log-normalisation; N2: analytic Pearson residuals. **Pre-registered:** RPE1, weak tertile, D1–D3 each required in both N1 and N2, with fewer than 10 qualifying T targets reported as un